# Marker PDF Converter + NVIDIA API (Google Colab)

Convierte PDF, imágenes, DOCX, PPTX, XLSX, HTML y EPUB a Markdown/HTML/JSON con [Marker](https://github.com/VikParuchuri/marker), usando la API gratuita de NVIDIA como LLM opcional. El resultado se descarga en un ZIP listo para descomprimir en tu vault de [Obsidian](https://obsidian.md).

**Ejecuta las celdas en orden, de arriba a abajo.** En el paso 5 eliges si conviertes **un documento o varios**: basta con seleccionar uno o varios archivos en el diálogo de subida. Si el runtime se reinicia, vuelve a empezar desde el paso 2.

Antes de empezar: *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*. Si vas a usar el LLM, consigue tu API key en [build.nvidia.com](https://build.nvidia.com) (empieza por `nvapi-`).

## 1. Comprobar GPU

In [ ]:
!nvidia-smi

## 2. Instalar Marker (versión fijada)

Se instala `marker-pdf 1.10.2`. **No uses la 2.0**: sus modelos de layout necesitan Docker (vLLM) o llama.cpp, que no existen en Colab, y da el error `docker binary not found`. Tarda unos minutos.

In [ ]:
!pip install -q "marker-pdf[full]==1.10.2" "surya-ocr>=0.17.1,<0.18" openai

from importlib.metadata import version
mv, sv = version("marker-pdf"), version("surya-ocr")
print("marker-pdf:", mv, "| surya-ocr:", sv)
assert mv.startswith("1."), "Se instaló una versión incorrecta de marker-pdf"
print("✓ Marker instalado")

## 3. Opciones

| Opción | Qué hace |
|---|---|
| `USE_LLM` | Usa el modelo de NVIDIA para mejorar tablas, fórmulas y texto. Más lento y necesita API key. |
| `NVIDIA_MODEL` | Modelo multimodal (texto + imagen) de build.nvidia.com. |
| `OUTPUT_FORMAT` | `markdown` genera el paquete para Obsidian; `html` y `json` se descargan tal cual los crea Marker. |
| `FORCE_OCR` | Actívalo solo si el texto sale corrupto o el PDF es escaneado. |
| `PAGE_RANGE` | Páginas a convertir, empezando en 0. Ej: `0-10` o `0,5-10`. Vacío = todo. Se aplica a cada documento. |
| `REDO_INLINE_MATH` | Rehace las fórmulas dentro del texto (solo con `USE_LLM`). |

In [ ]:
USE_LLM          = False                  # @param {type:"boolean"}
NVIDIA_MODEL     = "z-ai/glm-5.3-flash"   # @param {type:"string"}
OUTPUT_FORMAT    = "markdown"             # @param ["markdown", "html", "json"]
FORCE_OCR        = False                  # @param {type:"boolean"}
PAGE_RANGE       = ""                     # @param {type:"string"}
REDO_INLINE_MATH = True                   # @param {type:"boolean"}

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
PAGE_RANGE = PAGE_RANGE.strip()

print(f"LLM: {'sí (' + NVIDIA_MODEL + ')' if USE_LLM else 'no'} | formato: {OUTPUT_FORMAT} | "
      f"OCR forzado: {'sí' if FORCE_OCR else 'no'} | páginas: {PAGE_RANGE or 'todas'}")

## 4. Conectar con la API de NVIDIA (solo si `USE_LLM = True`)

Lee la key del secreto de Colab `NVIDIA_API_KEY` (icono de la llave en la barra lateral) o te la pide. Después prueba el modelo con texto y con una imagen. Sin LLM, esta celda no hace nada.

In [ ]:
from getpass import getpass

NVIDIA_API_KEY = ""

if not USE_LLM:
    print("Modo sin LLM: no hace falta API key. Pasa al paso 5.")
else:
    try:
        from google.colab import userdata
        NVIDIA_API_KEY = (userdata.get("NVIDIA_API_KEY") or "").strip()
        print("✓ Key leída del secreto de Colab NVIDIA_API_KEY")
    except Exception:
        pass
    if not NVIDIA_API_KEY:
        NVIDIA_API_KEY = getpass("Pega tu NVIDIA API key (nvapi-...): ").strip()

    import base64, io
    from openai import OpenAI
    from PIL import Image, ImageDraw

    client = OpenAI(base_url=NVIDIA_BASE_URL, api_key=NVIDIA_API_KEY)
    try:
        r = client.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[{"role": "user", "content": "Responde solo: OK"}],
            max_tokens=300,
        )
        print("✓ Texto OK. Respuesta:", (r.choices[0].message.content or "").strip()[:80])

        img = Image.new("RGB", (300, 100), "white")
        ImageDraw.Draw(img).text((20, 40), "HOLA 42", fill="black")
        buf = io.BytesIO(); img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        r = client.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": "¿Qué texto aparece en la imagen? Responde solo con el texto."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ]}],
            max_tokens=500,
        )
        print("✓ Visión OK. Respuesta:", (r.choices[0].message.content or "").strip()[:80])
    except Exception as e:
        print("❌ Falló la prueba de la API:", e)
        print("\n- 401/403: key incorrecta o con espacios")
        print("- 404: el modelo no existe (abajo tienes la lista si se pudo obtener)")
        print("- 429: límite gratuito alcanzado, espera un poco")
        if "404" in str(e):
            try:
                ids = sorted(m.id for m in client.models.list().data)
                print("\nModelos disponibles:\n" + "\n".join(ids))
            except Exception as e2:
                print("No se pudo listar los modelos:", e2)

## 5. Subir uno o varios documentos

Selecciona **un archivo** para convertir solo ese, o **varios** (Ctrl o Shift en el diálogo) para convertirlos todos en lote. Volver a ejecutar esta celda descarta lo subido antes.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

INPUT_DIR = Path("/content/input")
shutil.rmtree(INPUT_DIR, ignore_errors=True)
INPUT_DIR.mkdir(parents=True)

uploaded = files.upload()
assert uploaded, "No se subió ningún archivo"

# Marker guarda cada documento en una carpeta con su nombre sin extensión,
# así que "tema.pdf" y "tema.docx" tienen que llamarse distinto.
input_files, used_stems = [], set()
for name in uploaded:
    src = Path(name)
    stem, n = src.stem, 2
    while stem.lower() in used_stems:
        stem, n = f"{src.stem}_{n}", n + 1
    used_stems.add(stem.lower())
    dest = INPUT_DIR / f"{stem}{src.suffix}"
    shutil.move(str(src), str(dest))
    input_files.append(dest)

modo = "un documento" if len(input_files) == 1 else f"{len(input_files)} documentos en lote"
print(f"\n✓ Modo: {modo}")
for f in input_files:
    print("  -", f.name)

## 6. Convertir

Con un documento usa `marker_single`; con varios, `marker`, que carga los modelos una sola vez para todos. Al final indica qué documentos fallaron, si alguno lo hizo.

In [ ]:
import shutil, subprocess
from importlib.metadata import version
from pathlib import Path

assert globals().get("input_files"), "Primero sube los documentos (paso 5)"
assert version("marker-pdf").startswith("1."), "marker-pdf 2.x necesita Docker. Vuelve a ejecutar el paso 2."
NVIDIA_API_KEY = globals().get("NVIDIA_API_KEY", "")

OUTPUT_DIR = Path("/content/output")
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
OUTPUT_DIR.mkdir(parents=True)

if len(input_files) == 1:
    cmd = ["marker_single", str(input_files[0])]
else:
    # Con más de un worker la T4 se queda sin memoria
    cmd = ["marker", str(INPUT_DIR), "--workers", "1"]

cmd += ["--output_format", OUTPUT_FORMAT, "--output_dir", str(OUTPUT_DIR)]
if FORCE_OCR:
    cmd.append("--force_ocr")
if PAGE_RANGE:
    cmd += ["--page_range", PAGE_RANGE]
if USE_LLM:
    assert NVIDIA_API_KEY, "Falta la API key: ejecuta el paso 4"
    cmd += ["--use_llm",
            "--llm_service", "marker.services.openai.OpenAIService",
            "--openai_base_url", NVIDIA_BASE_URL,
            "--openai_model", NVIDIA_MODEL,
            "--openai_api_key", NVIDIA_API_KEY]
    if REDO_INLINE_MATH:
        cmd.append("--redo_inline_math")

print("Ejecutando:", " ".join("***" if NVIDIA_API_KEY and a == NVIDIA_API_KEY else a for a in cmd), "\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()

# En lote, Marker sigue con el resto si un documento falla y termina con código 0
out_ext = {"markdown": "md", "html": "html", "json": "json"}[OUTPUT_FORMAT]
converted = [f for f in input_files if (OUTPUT_DIR / f.stem / f"{f.stem}.{out_ext}").exists()]
failed = [f for f in input_files if f not in converted]

print()
if converted:
    print(f"✓ Convertidos: {len(converted)} de {len(input_files)}")
if failed:
    print(f"❌ Fallaron ({len(failed)}): " + ", ".join(f.name for f in failed))
    print("   Busca su nombre en el log de arriba para ver el error.")
if proc.returncode != 0:
    print(f"❌ Marker terminó con error (código {proc.returncode}). Revisa el log de arriba.")

## 7. Empaquetar y descargar

Genera un ZIP con una carpeta por documento: `nombre/nombre.md` y `nombre/images/`. Los enlaces de imagen se reescriben a `images/...` y las fórmulas quedan en `$...$` (inline) y `$$...$$` (bloque). Descomprime el ZIP dentro de tu vault de Obsidian.

Con `html` o `json`, el ZIP contiene la salida de Marker sin modificar. Puedes volver a ejecutar esta celda sin convertir de nuevo.

In [ ]:
import re, shutil, urllib.parse
from pathlib import Path
from google.colab import files

assert globals().get("converted"), "No hay documentos convertidos. Ejecuta antes la conversión (paso 6)."

IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".gif"}
MD_IMAGE = re.compile(r"!\[([^\]]*)\]\(([^)]+)\)")


def obsidian_name(stem):
    return re.sub(r'[\\/:*?"<>|#^\[\]]', "_", stem).strip() or "documento"


def to_obsidian(md_path, dest):
    # Devuelve (texto, nº de imágenes, enlaces rotos)
    src_dir = md_path.parent
    imgs = {f.name: f for f in src_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
    dest.mkdir(parents=True)
    if imgs:
        (dest / "images").mkdir()
    for name, f in imgs.items():
        shutil.copy(f, dest / "images" / name)

    rotos = []
    def fix_img(m):
        alt, target = m.group(1), urllib.parse.unquote(m.group(2).strip())
        name = Path(target).name
        if name in imgs:
            return f"![{alt}](images/{urllib.parse.quote(name)})"
        rotos.append(target)
        return m.group(0)

    text = md_path.read_text(encoding="utf-8")
    text = MD_IMAGE.sub(fix_img, text)
    # Fórmulas: Obsidian solo entiende $...$ y $$...$$
    text = re.sub(r"\\\[(.+?)\\\]", lambda m: "\n$$" + m.group(1).strip() + "$$\n", text, flags=re.S)
    text = re.sub(r"\\\((.+?)\\\)", lambda m: "$" + m.group(1).strip() + "$", text, flags=re.S)
    text = re.sub(r'<span id="[^"]*"></span>', "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip() + "\n"

    (dest / f"{dest.name}.md").write_text(text, encoding="utf-8")
    return text, len(imgs), rotos


BASE = Path("/content/package")
shutil.rmtree(BASE, ignore_errors=True)
BASE.mkdir(parents=True)

used_names = set()
for f in converted:
    nombre, n = obsidian_name(f.stem), 2
    base_nombre = nombre
    while nombre.lower() in used_names:
        nombre, n = f"{base_nombre}_{n}", n + 1
    used_names.add(nombre.lower())
    dest = BASE / nombre

    if OUTPUT_FORMAT != "markdown":
        shutil.copytree(OUTPUT_DIR / f.stem, dest)
        print(f"✓ {nombre}/ ({OUTPUT_FORMAT})")
        continue

    text, n_imgs, rotos = to_obsidian(OUTPUT_DIR / f.stem / f"{f.stem}.md", dest)
    n_block = len(re.findall(r"\$\$", text)) // 2
    n_inline = len(re.findall(r"(?<!\$)\$(?!\$)[^$\n]+\$(?!\$)", text))
    print(f"✓ {nombre}/{nombre}.md  ({len(text):,} caracteres, {n_imgs} imágenes, "
          f"{n_block} fórmulas en bloque y ~{n_inline} inline)")
    if rotos:
        print(f"  ⚠️ {len(rotos)} enlace(s) a imágenes que no existen: {rotos[:5]}")
    if not n_imgs:
        print("  ⚠️ Marker no extrajo ninguna imagen (¿PAGE_RANGE con pocas páginas, o sin figuras detectables?)")

suffix = "obsidian" if OUTPUT_FORMAT == "markdown" else OUTPUT_FORMAT
zip_name = f"{nombre}_{suffix}" if len(converted) == 1 else f"documentos_{suffix}"
zip_path = shutil.make_archive(f"/content/{zip_name}", "zip", root_dir=BASE)
print(f"\n📦 {Path(zip_path).name}: {len(converted)} documento(s)")
files.download(zip_path)